[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/01-python-basics/02_python_for_ml_idioms.ipynb)

# Python Idioms an ML Course Assumes

**Session 2 · companion to HW 1 · nothing here is a homework answer**

The Python refresher notebook covers the language. This one covers the handful
of behaviours that, left unexamined, produce bugs you will spend an afternoon
on later in this course: aliasing, mutable defaults, laziness, and the class
shape that every scikit-learn object wears.

Predict each output before you run the cell. The value of this notebook is in
the gap between what you predicted and what printed.

In [1]:
a = [1, 2, 3]
b = a                 # not a copy: another name for the same list
c = a[:]              # a copy

a.append(4)
print("a:", a)
print("b:", b, "   <- changed, because b is a")
print("c:", c, "   <- unchanged, because c is a copy")
print("a is b:", a is b, "  a is c:", a is c)

a: [1, 2, 3, 4]
b: [1, 2, 3, 4]    <- changed, because b is a
c: [1, 2, 3]    <- unchanged, because c is a copy
a is b: True   a is c: False


`b = a` binds a second name to one object. Nothing was copied, so nothing could
be left behind. This is the mechanism behind the most common surprise in pandas
and NumPy too: a slice may be a view of the original, and writing to it writes
through.

The rule that saves you: **assignment never copies.** If you want a copy, ask
for one — `list(x)`, `x.copy()`, `copy.deepcopy(x)` for nested structures.

In [2]:
import copy

nested = [[1, 2], [3, 4]]
shallow = nested.copy()
deep = copy.deepcopy(nested)

nested[0].append(99)
print("shallow:", shallow, "  <- the inner list is still shared")
print("deep:   ", deep, "  <- fully independent")

shallow: [[1, 2, 99], [3, 4]]   <- the inner list is still shared
deep:    [[1, 2], [3, 4]]   <- fully independent


## The mutable default

A default argument is evaluated **once**, when the function is defined — not
once per call. A mutable default therefore becomes shared state between calls,
which looks like the function remembering things it was never asked to.

In [3]:
def collect_broken(value, into=[]):        # the bug
    into.append(value)
    return into

print(collect_broken(1))
print(collect_broken(2), "  <- 1 is still there")
print(collect_broken(3), "  <- and so is 2")
print("the default itself:", collect_broken.__defaults__)

[1]
[1, 2]   <- 1 is still there
[1, 2, 3]   <- and so is 2
the default itself: ([1, 2, 3],)


In [4]:
def collect_fixed(value, into=None):       # the fix
    if into is None:
        into = []
    into.append(value)
    return into

print(collect_fixed(1))
print(collect_fixed(2))
print(collect_fixed(3), "  <- a fresh list every call")

[1]
[2]
[3]   <- a fresh list every call


You will meet this exact shape in library code: `random_state=None`,
`categories=None`, `sample_weight=None`. `None` as a default and the real value
built inside is not a stylistic preference, it is the only correct way to give a
mutable default.

## Comprehensions, and when to stop

A comprehension is a loop whose result is the point. Use one when you are
building a collection; use a `for` statement when you are doing something.

In [5]:
readings = [12.5, 14.0, None, 15.5, None, 11.0]

clean = [r for r in readings if r is not None]
scaled = {i: round(r / max(clean), 3) for i, r in enumerate(clean)}
above = {r for r in clean if r > 12.0}

print("clean :", clean)
print("scaled:", scaled)
print("above :", sorted(above))

clean : [12.5, 14.0, 15.5, 11.0]
scaled: {0: 0.806, 1: 0.903, 2: 1.0, 3: 0.71}
above : [12.5, 14.0, 15.5]


Three forms, one syntax: brackets give a list, braces with `key: value` give a
dict, braces without give a set. When a comprehension needs a comment to be
readable, it has stopped being an improvement over the loop.

## Generators: the same values, without the list

A generator computes on demand. For a course that will hand you datasets larger
than memory, the distinction is practical rather than academic.

In [6]:
import sys

n = 1_000_000
as_list = [i * i for i in range(n)]
as_generator = (i * i for i in range(n))

print(f"list      {sys.getsizeof(as_list):>10,} bytes")
print(f"generator {sys.getsizeof(as_generator):>10,} bytes")
print("same first five:", as_list[:5], "vs", [next(as_generator) for _ in range(5)])

list       8,448,728 bytes
generator        200 bytes
same first five: [0, 1, 4, 9, 16] vs [0, 1, 4, 9, 16]


The generator holds a recipe, not a million integers. The cost is that it is
consumed as you read it — a generator can be iterated once, and `len()` does not
work on it. That trade is why `DataLoader` in Session 20 hands you batches from
an iterator rather than a list.

## The class behind `fit` and `predict`

Every scikit-learn estimator is this shape: a constructor that stores
hyperparameters, a `fit` that learns state from data and returns `self`, and a
`predict` that uses that state. The baseline below is deliberately trivial — it
predicts the training mean whatever you give it — because the point is the
protocol, not the model.

In [7]:
import numpy as np

class MeanBaseline:
    """Predicts the training mean. Useless as a model, correct as a shape."""

    def __init__(self, clip_negative=False):
        self.clip_negative = clip_negative       # a hyperparameter: set, not learned

    def fit(self, X, y):
        self.value_ = float(np.mean(y))          # learned state: trailing underscore
        self.n_features_in_ = np.asarray(X).shape[1]
        return self                              # so you can chain .fit(X, y).predict(X)

    def predict(self, X):
        if not hasattr(self, "value_"):
            raise RuntimeError("call fit before predict")
        out = np.full(len(X), self.value_)
        return np.clip(out, 0, None) if self.clip_negative else out

X = np.arange(20).reshape(10, 2)
y = np.array([3.0, 5.0, 4.0, 8.0, 6.0, 7.0, 5.5, 4.5, 9.0, 2.0])

model = MeanBaseline().fit(X, y)
print("learned state:", {k: v for k, v in vars(model).items() if k.endswith("_")})
print("predictions  :", model.predict(X[:3]))

learned state: {'value_': 5.4, 'n_features_in_': 2}
predictions  : [5.4 5.4 5.4]


Two conventions in that class are worth copying, because scikit-learn relies on
them and so will your own code:

- **`self.value_` with a trailing underscore** marks state that came from data.
  Anything without one was set by the caller. `vars(model)` above therefore
  reads as a summary of what fitting learned.
- **`fit` returns `self`**, which is what makes `Pipeline` possible at all.

And the failure mode is worth seeing once:

In [8]:
try:
    MeanBaseline().predict(X)
except RuntimeError as err:
    print("RuntimeError:", err)

RuntimeError: call fit before predict


## Asking forgiveness, not permission

Python's convention is to attempt the operation and handle the failure, rather
than to check every precondition first. The checking version is longer, slower,
and still wrong when the state changes between the check and the use.

In [9]:
record = {"id": 7, "value": "12.5"}

# Look before you leap
if "value" in record and record["value"].replace(".", "", 1).isdigit():
    lbyl = float(record["value"])
else:
    lbyl = float("nan")

# Easier to ask forgiveness than permission
try:
    eafp = float(record["value"])
except (KeyError, ValueError, TypeError):
    eafp = float("nan")

print(lbyl, eafp)

12.5 12.5


Catch the exceptions you expect, not `Exception`. A bare `except:` also catches
the typo in your own code and turns a five-second fix into an afternoon.

## Paths are objects

`pathlib` removes an entire category of string-joining bugs, and it is what the
course's own scripts use.

In [10]:
from pathlib import Path

data_dir = Path("data") / "raw"
target = data_dir / "readings.csv"

print("as text     :", target)
print("suffix      :", target.suffix)
print("stem        :", target.stem)
print("parent      :", target.parent)
print("exists      :", target.exists())
print("sibling     :", target.with_name("readings_clean.csv"))

as text     : data/raw/readings.csv
suffix      : .csv
stem        : readings
parent      : data/raw
exists      : False
sibling     : data/raw/readings_clean.csv


`Path("data") / "raw"` is correct on Windows and macOS alike, and
`target.with_suffix(".parquet")` says what it means where
`name.replace(".csv", ".parquet")` quietly mangles a file called
`2026.csv.backup.csv`.

## What to take from this

- Assignment binds a name; it never copies. Copy explicitly when you mean to.
- `None` is the only safe default for a mutable argument.
- A comprehension builds a collection; a loop does work.
- A generator is a recipe you can read once.
- `fit` learns state and returns `self`; learned state carries a trailing
  underscore. Every estimator in this course follows that contract.

## Where to go next

- **Reading, Session 2** — the same material with the language semantics
  spelled out.
- **HW 1** — uses all of it on real tables. The functions it asks you to write
  are deliberately absent here.